In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings

warnings.filterwarnings('ignore')

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [2]:
# Load data
print("Loading data...")
train_df = pd.read_csv('/kaggle/input/trademaster25/train_v2.csv')
test_df = pd.read_csv('/kaggle/input/trademaster25/test_v2.csv')

print(f"Training set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Feature and target columns
feature_cols = [f'feature_{i}' for i in range(1, 31)]
target_cols = ['target_short', 'target_medium', 'target_long']
time_cols = ['date_id', 'minute_id']
TARGET_WEIGHTS = {'short': 0.5, 'medium': 0.3, 'long': 0.2}

print(f"Features: {len(feature_cols)}")
print(f"Targets: {target_cols}")
print(f"Weights: {TARGET_WEIGHTS}")

train_df.describe()

Loading data...
Training set: (139392, 37)
Test set: (34348, 34)
Features: 30
Targets: ['target_short', 'target_medium', 'target_long']
Weights: {'short': 0.5, 'medium': 0.3, 'long': 0.2}


,id,date_id,minute_id,stock_id,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,...,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,target_short,target_medium,target_long
count,139392.000000,139392.000000,139392.000000,139392.0,139392.000000,1.393910e+05,139382.000000,139379.000000,139362.000000,139392.000000,...,139373.000000,139358.000000,139363.000000,139392.000000,139391.000000,139362.000000,139362.000000,139392.000000,139392.000000,139392.000000
mean,69695.500000,289.900138,119.466942,0.0,0.100651,-inf,-0.002015,0.067555,0.137022,-0.004771,...,0.010554,-52.780710,0.004630,0.399447,-0.000003,0.000934,-0.000098,-0.000033,-0.000192,-0.000766
std,40239.148699,167.663109,69.270219,0.0,0.300868,NaN,0.226708,0.042766,0.964826,1.020697,...,0.009089,31.683331,1.090614,0.230623,0.001940,0.000715,0.010186,0.005871,0.014408,0.028882
min,0.000000,0.000000,0.000000,0.0,0.000000,-inf,-5.270000,0.000000,-5.477226,-1.904991,...,0.000000,-100.000000,-3.648028,0.000000,-0.093234,0.000000,-0.090227,-0.091812,-0.086804,-0.131662
25%,34847.750000,145.000000,59.000000,0.0,0.000000,-3.201773e-04,-0.090000,0.039286,-0.264227,-0.803271,...,0.005339,-81.395349,-0.701752,0.199723,-0.000807,0.000552,-0.004991,-0.002650,-0.007663,-0.018639
50%,69695.500000,290.000000,119.000000,0.0,0.000000,0.000000e+00,-0.010000,0.057143,0.160343,0.031092,...,0.008077,-54.726912,-0.282123,0.399447,0.000000,0.000764,-0.000569,-0.000275,-0.000962,-0.003101
75%,104543.250000,435.000000,179.000000,0.0,0.000000,3.548881e-04,0.080000,0.082143,0.568293,0.811304,...,0.012601,-25.000000,0.418252,0.599170,0.000712,0.001073,0.004059,0.002328,0.006019,0.015318
max,139391.000000,580.000000,239.000000,0.0,1.000000,4.886260e+00,4.370000,0.774286,5.477226,1.904696,...,0.157762,-0.000000,5.293741,0.798894,0.060000,0.016995,0.101807,0.095770,0.095640,0.164725


In [3]:
# Fill NaN with median and clip extreme values
for col in feature_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    test_df[col].fillna(median_val, inplace=True)

    # Clip extreme values at 1% and 99% quantiles
    lower = train_df[col].quantile(0.01)
    upper = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(lower, upper)
    test_df[col] = test_df[col].clip(lower, upper)

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# # 可视化第一个特征的分布变化
# plt.figure(figsize=(10, 4))
# sns.histplot(train_df['feature_1'], kde=True)
# plt.title('Feature 1 (Before GaussRank)')
# plt.show()

from sklearn.preprocessing import QuantileTransformer

print("4.1.2 GaussRank 归一化 (Rank-Gauss)")
print("=" * 80)

# 使用 QuantileTransformer 进行 GaussRank 归一化
# output_distribution='normal' 将数据映射为高斯分布
gauss_scaler = QuantileTransformer(output_distribution='normal', random_state=42)

print("Applying GaussRank normalization...")
# 对特征列进行转换
train_df[feature_cols] = gauss_scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = gauss_scaler.transform(test_df[feature_cols])

print("GaussRank normalization complete.")

# 验证归一化后的分布
print("\n归一化后的偏度和峰度 (前5个特征):")
print(train_df[feature_cols[:5]].agg(['skew', 'kurtosis']))

# # 可视化第一个特征的分布变化
# plt.figure(figsize=(10, 4))
# sns.histplot(train_df['feature_1'], kde=True)
# plt.title('Feature 1 (After GaussRank)')
# plt.show()

4.1.2 GaussRank 归一化 (Rank-Gauss)
Applying GaussRank normalization...
GaussRank normalization complete.

归一化后的偏度和峰度 (前5个特征):
          feature_1  feature_2  feature_3  feature_4  feature_5
skew       2.654684  -0.016043  -0.000495   0.021033  -0.001348
kurtosis   5.047422   5.424841   5.448460   5.543992   5.352595


In [5]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import pickle

print("4.1.3 Purged Group Time Series Split (核心验证策略)")
print("=" * 80)

class PurgedGroupTimeSeriesSplit:
    def __init__(self, n_splits=5, gap=0):
        self.n_splits = n_splits
        self.gap = gap

    def split(self, X, y=None, groups=None):
        if groups is None:
            tscv = TimeSeriesSplit(n_splits=self.n_splits)
            for train_idx, test_idx in tscv.split(X):
                test_start = test_idx[0]
                train_end_limit = test_start - self.gap
                train_idx_purged = train_idx[train_idx <= train_end_limit]
                if len(train_idx_purged) > 0:
                    yield train_idx_purged, test_idx
        else:
            unique_groups = np.unique(groups)
            unique_groups.sort()
            tscv = TimeSeriesSplit(n_splits=self.n_splits)
            for train_groups_idx, test_groups_idx in tscv.split(unique_groups):
                train_groups = unique_groups[train_groups_idx]
                test_groups = unique_groups[test_groups_idx]
                
                test_start_group_idx = test_groups_idx[0]
                train_end_group_idx_limit = test_start_group_idx - self.gap
                train_groups_idx_purged = train_groups_idx[train_groups_idx <= train_end_group_idx_limit]
                
                if len(train_groups_idx_purged) > 0:
                    train_groups_purged = unique_groups[train_groups_idx_purged]
                    train_idx = np.where(np.isin(groups, train_groups_purged))[0]
                    test_idx = np.where(np.isin(groups, test_groups))[0]
                    yield train_idx, test_idx

# --- 策略配置 ---
# 1. Grouping: 使用 date_id (天)
#    理由: 即使有 minutes_id，按天分组能更彻底地防止日内微观结构噪音的泄露，且计算效率更高。
# 2. Gap: 取决于预测目标的最长跨度
#    理由: 如果同时预测"几分钟后"(Short)和"几天后"(Long)，必须使用覆盖 Long 的 Gap。
#    例如: Long 是预测 5 天后的收益率，那么 Gap 至少要设为 5 (天)。
#    虽然这对 Short 目标来说 Gap 过大（浪费了一些近期数据），但能确保验证集的绝对安全（无泄露）。

N_SPLITS = 5
# 假设 date_id 是连续的整数代表天数
# 假设 target_long 预测未来 5 天，则设置 GAP = 5 + 1 (缓冲) = 6
GAP = 6 

# 检查并设置 Groups
groups = None
if 'date_id' in train_df.columns:
    groups = train_df['date_id'].values
    print(f"✅ 使用 'date_id' 进行分组 (Groups: {len(np.unique(groups))} days)")
    print(f"✅ 设置 Gap = {GAP} (days) 以覆盖最长预测窗口")
else:
    print("no 'date_id' column found for grouping. Using standard TimeSeriesSplit.")

cv = PurgedGroupTimeSeriesSplit(n_splits=N_SPLITS, gap=GAP)

# 生成并保存分割
splits = []
for train_idx, test_idx in cv.split(train_df, groups=groups):
    splits.append((train_idx, test_idx))

with open('cv_splits.pkl', 'wb') as f:
    pickle.dump(splits, f)

print(f"\n已生成 {len(splits)} 折交叉验证分割。")
print(f"分割方案已保存至: cv_splits.pkl")

# 打印第一折的详细信息用于检查
if len(splits) > 0:
    train_idx_0, test_idx_0 = splits[0]
    print("\n--- Fold 1 详情 ---")
    print(f"Train samples: {len(train_idx_0):,}")
    print(f"Test samples:  {len(test_idx_0):,}")
    if groups is not None:
        print(f"Train dates: {groups[train_idx_0].min()} -> {groups[train_idx_0].max()}")
        print(f"Test dates:  {groups[test_idx_0].min()} -> {groups[test_idx_0].max()}")
        print(f"Gap 验证: Test Start ({groups[test_idx_0].min()}) - Train End ({groups[train_idx_0].max()}) = {groups[test_idx_0].min() - groups[train_idx_0].max()}")

4.1.3 Purged Group Time Series Split (核心验证策略)
✅ 使用 'date_id' 进行分组 (Groups: 581 days)
✅ 设置 Gap = 6 (days) 以覆盖最长预测窗口

已生成 5 折交叉验证分割。
分割方案已保存至: cv_splits.pkl

--- Fold 1 详情 ---
Train samples: 23,040
Test samples:  23,040
Train dates: 0 -> 95
Test dates:  101 -> 196
Gap 验证: Test Start (101) - Train End (95) = 6


In [6]:
import cudf
import cuml
from cuml.neighbors import NearestNeighbors
from cuml.preprocessing import StandardScaler as cuStandardScaler
import numpy as np
import pandas as pd
import gc
import pickle

print("4.2 阶段二：RAPIDS 加速的特征工程 (Train + Test)")
print("=" * 80)

# 准备数据 (转换为 GPU DataFrame)
print("Preparing data for GPU...")
X_all = train_df[feature_cols].values
y_all = train_df[target_cols].values
X_test_all = test_df[feature_cols].values # Test set

# 定义 KNN 参数
K_VALUES = [5, 10, 20]
METRICS = ['euclidean', 'cosine']

# 初始化结果容器
n_samples = len(train_df)
n_test_samples = len(test_df)
n_knn_features = len(K_VALUES) * len(METRICS) * (1 + len(target_cols)) 
knn_feature_names = []
knn_features = np.zeros((n_samples, n_knn_features), dtype=np.float32)
knn_features_test = np.zeros((n_test_samples, n_knn_features), dtype=np.float32)

print(f"Generating {n_knn_features} KNN features...")

# 加载之前保存的 splits
with open('cv_splits.pkl', 'rb') as f:
    splits = pickle.load(f)

knn_features[:] = np.nan

# --- Part 1: Generate KNN features for Train (CV) ---
for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n--- Processing Fold {fold + 1}/{len(splits)} (Train CV) ---")
    
    # 数据移至 GPU
    X_train_gpu = cudf.DataFrame(X_all[train_idx], columns=feature_cols)
    X_val_gpu = cudf.DataFrame(X_all[val_idx], columns=feature_cols)
    
    # 标准化 (Fit on Train, Transform on Val)
    scaler = cuStandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_gpu)
    X_val_scaled = scaler.transform(X_val_gpu)
    
    col_idx = 0
    
    for k in K_VALUES:
        for metric in METRICS:
            # Build KNN model
            model = NearestNeighbors(n_neighbors=k, metric=metric)
            model.fit(X_train_scaled)
            
            # Get distances and indices
            distances, indices = model.kneighbors(X_val_scaled)
            
            # 1. Feature: Mean Distance
            dist_mean = distances.mean(axis=1)
            
            feat_name_dist = f'knn_k{k}_{metric}_dist_mean'
            if fold == 0: knn_feature_names.append(feat_name_dist)
            
            knn_features[val_idx, col_idx] = dist_mean.to_numpy()
            col_idx += 1
            
            # 2. Feature: Mean Target of Neighbors
            indices_cpu = indices.to_numpy()
            y_train_cpu = y_all[train_idx] 
            
            for t_i, target_col in enumerate(target_cols):
                neighbor_targets = y_train_cpu[indices_cpu, t_i]
                target_mean = neighbor_targets.mean(axis=1)
                
                feat_name_target = f'knn_k{k}_{metric}_{target_col}_mean'
                if fold == 0: knn_feature_names.append(feat_name_target)
                
                knn_features[val_idx, col_idx] = target_mean
                col_idx += 1
            
    del X_train_gpu, X_val_gpu, X_train_scaled, X_val_scaled
    gc.collect()

# --- Part 2: Generate KNN features for Test (Full Train) ---
print("\n--- Processing Test Set (Full Train Context) ---")
X_train_full_gpu = cudf.DataFrame(X_all, columns=feature_cols)
X_test_gpu = cudf.DataFrame(X_test_all, columns=feature_cols)

scaler_full = cuStandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full_gpu)
X_test_scaled = scaler_full.transform(X_test_gpu)

col_idx = 0
for k in K_VALUES:
    for metric in METRICS:
        model = NearestNeighbors(n_neighbors=k, metric=metric)
        model.fit(X_train_full_scaled)
        
        distances, indices = model.kneighbors(X_test_scaled)
        
        # 1. Dist Mean
        dist_mean = distances.mean(axis=1)
        knn_features_test[:, col_idx] = dist_mean.to_numpy()
        col_idx += 1
        
        # 2. Target Mean
        indices_cpu = indices.to_numpy()
        y_train_cpu = y_all # Full labels
        
        for t_i, target_col in enumerate(target_cols):
            neighbor_targets = y_train_cpu[indices_cpu, t_i]
            target_mean = neighbor_targets.mean(axis=1)
            knn_features_test[:, col_idx] = target_mean
            col_idx += 1

del X_train_full_gpu, X_test_gpu, X_train_full_scaled, X_test_scaled
gc.collect()

# 保存 KNN 特征
knn_df = pd.DataFrame(knn_features, columns=knn_feature_names)
knn_df = knn_df.fillna(knn_df.mean())

knn_test_df = pd.DataFrame(knn_features_test, columns=knn_feature_names)
knn_test_df = knn_test_df.fillna(knn_test_df.mean()) # Should not have NaNs usually

print(f"\nKNN Features Shape (Train): {knn_df.shape}")
print(f"KNN Features Shape (Test): {knn_test_df.shape}")

knn_df.to_parquet('knn_features.parquet')
knn_test_df.to_parquet('knn_features_test.parquet')
print("✅ KNN features saved to 'knn_features.parquet' and 'knn_features_test.parquet'")

# 合并回主 DataFrame (Train only for DAE training, but we need Test for DAE inference)
train_df_with_knn = pd.concat([train_df, knn_df], axis=1)
test_df_with_knn = pd.concat([test_df, knn_test_df], axis=1)
all_feature_cols = feature_cols + knn_feature_names
print(f"Total features for DAE: {len(all_feature_cols)}")

4.2 阶段二：RAPIDS 加速的特征工程 (Train + Test)
Preparing data for GPU...
Generating 24 KNN features...

--- Processing Fold 1/5 (Train CV) ---

--- Processing Fold 2/5 (Train CV) ---

--- Processing Fold 3/5 (Train CV) ---

--- Processing Fold 4/5 (Train CV) ---

--- Processing Fold 5/5 (Train CV) ---

--- Processing Test Set (Full Train Context) ---

KNN Features Shape (Train): (139392, 24)
KNN Features Shape (Test): (34348, 24)
✅ KNN features saved to 'knn_features.parquet' and 'knn_features_test.parquet'
Total features for DAE: 54


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

print("4.3 阶段三：DAE 潜在特征提取 (Train + Test)")
print("=" * 80)

# DAE Dataset with Swap Noise
class DAEDataset(Dataset):
    def __init__(self, X, noise_prob=0.15, training=True):
        self.X = torch.FloatTensor(X)
        self.noise_prob = noise_prob
        self.n_features = X.shape[1]
        self.training = training
        
    def train(self):
        self.training = True
        
    def eval(self):
        self.training = False
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        
        # Swap Noise
        if self.noise_prob > 0 and self.training:
            mask = torch.rand(self.n_features) < self.noise_prob
            if mask.any():
                noise_idx = torch.randint(0, len(self.X), (1,)).item()
                x[mask] = self.X[noise_idx][mask]
                
        return x, self.X[idx] 

# Bottleneck DAE Model
class DAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, latent_dim=64):
        super(DAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, latent_dim),
            nn.BatchNorm1d(latent_dim), 
            nn.SiLU() 
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Linear(128, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim)
        )
        
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# 准备数据
X_dae = train_df_with_knn[all_feature_cols].fillna(0).values
X_dae_test = test_df_with_knn[all_feature_cols].fillna(0).values # Test Data

print(f"DAE Input Shape (Train): {X_dae.shape}")
print(f"DAE Input Shape (Test): {X_dae_test.shape}")

# Dataloader
BATCH_SIZE = 512
train_dataset_dae = DAEDataset(X_dae, noise_prob=0.15, training=True)
train_loader_dae = DataLoader(train_dataset_dae, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# 初始化模型
INPUT_DIM = X_dae.shape[1]
dae_model = DAE(input_dim=INPUT_DIM).to(DEVICE)

# 训练配置
optimizer = optim.AdamW(dae_model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.MSELoss()
EPOCHS = 50 
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, steps_per_epoch=len(train_loader_dae), epochs=EPOCHS
)

print("\nStarting DAE Training...")
dae_model.train()

for epoch in range(EPOCHS):
    total_loss = 0
    for x_noisy, x_clean in train_loader_dae:
        x_noisy, x_clean = x_noisy.to(DEVICE), x_clean.to(DEVICE)
        
        optimizer.zero_grad()
        reconstructed, _ = dae_model(x_noisy)
        loss = criterion(reconstructed, x_clean)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader_dae)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.6f}")

print("DAE Training Complete.")

# 提取潜在特征 (Latent Features) - Train
print("\nExtracting Latent Features (Train)...")
dae_model.eval()
latent_features = []
extract_dataset = DAEDataset(X_dae, noise_prob=0.0, training=False)
extract_loader = DataLoader(extract_dataset, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader:
        x = x.to(DEVICE)
        _, latent = dae_model(x)
        latent_features.append(latent.cpu().numpy())

latent_features = np.vstack(latent_features)
dae_feat_cols = [f'dae_{i}' for i in range(latent_features.shape[1])]
dae_df = pd.DataFrame(latent_features, columns=dae_feat_cols)

# 提取潜在特征 (Latent Features) - Test
print("Extracting Latent Features (Test)...")
latent_features_test = []
extract_dataset_test = DAEDataset(X_dae_test, noise_prob=0.0, training=False)
extract_loader_test = DataLoader(extract_dataset_test, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for x, _ in extract_loader_test:
        x = x.to(DEVICE)
        _, latent = dae_model(x)
        latent_features_test.append(latent.cpu().numpy())

latent_features_test = np.vstack(latent_features_test)
dae_test_df = pd.DataFrame(latent_features_test, columns=dae_feat_cols)

print(f"Latent Features Shape (Train): {dae_df.shape}")
print(f"Latent Features Shape (Test): {dae_test_df.shape}")

dae_df.to_parquet('dae_features.parquet')
dae_test_df.to_parquet('dae_features_test.parquet')
print("✅ DAE features saved to 'dae_features.parquet' and 'dae_features_test.parquet'")

4.3 阶段三：DAE 潜在特征提取 (Train + Test)
DAE Input Shape (Train): (139392, 54)
DAE Input Shape (Test): (34348, 54)

Starting DAE Training...
Epoch 10/50, Loss: 0.328940
Epoch 20/50, Loss: 0.295459
Epoch 30/50, Loss: 0.275394
Epoch 40/50, Loss: 0.266507
Epoch 50/50, Loss: 0.261826
DAE Training Complete.

Extracting Latent Features (Train)...
Extracting Latent Features (Test)...
Latent Features Shape (Train): (139392, 64)
Latent Features Shape (Test): (34348, 64)
✅ DAE features saved to 'dae_features.parquet' and 'dae_features_test.parquet'


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print("4.4 阶段四：FT-Transformer 深度优化 (H100)")
print("=" * 80)

class ReGLU(nn.Module):
    def forward(self, x):
        a, b = x.chunk(2, dim=-1)
        return a * F.relu(b)

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_num_features, d_token))
        self.bias = nn.Parameter(torch.randn(n_num_features, d_token))
        
    def forward(self, x):
        # x: (batch, n_features)
        # out: (batch, n_features, d_token)
        x = x.unsqueeze(-1) * self.weight + self.bias
        return x

class FTTransformer(nn.Module):
    def __init__(
        self, 
        n_num_features, 
        n_targets, 
        d_token=192, 
        n_layers=3, 
        n_heads=8, 
        d_ffn_factor=1.33, 
        attention_dropout=0.2, 
        ffn_dropout=0.1,
        residual_dropout=0.0,
    ):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, d_token)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token))
        
        self.layers = nn.ModuleList([])
        # ReGLU input size is d_ffn * 2, output is d_ffn
        d_ffn = int(d_token * d_ffn_factor * 2 / 3) * 2 
        
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'norm1': nn.LayerNorm(d_token),
                'attn': nn.MultiheadAttention(d_token, n_heads, dropout=attention_dropout, batch_first=True),
                'norm2': nn.LayerNorm(d_token),
                'ffn': nn.Sequential(
                    nn.Linear(d_token, d_ffn * 2),
                    ReGLU(),
                    nn.Dropout(ffn_dropout),
                    nn.Linear(d_ffn, d_token),
                    nn.Dropout(residual_dropout),
                )
            }))
            
        self.head = nn.Sequential(
            nn.LayerNorm(d_token),
            nn.ReLU(),
            nn.Linear(d_token, n_targets)
        )
        
    def forward(self, x):
        # x: (batch, n_features)
        x = self.tokenizer(x) # (batch, n_features, d_token)
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        
        for layer in self.layers:
            # Pre-Norm Attention
            x_norm = layer['norm1'](x)
            attn_out, _ = layer['attn'](x_norm, x_norm, x_norm)
            x = x + attn_out
            
            # Pre-Norm FFN
            x_norm = layer['norm2'](x)
            ffn_out = layer['ffn'](x_norm)
            x = x + ffn_out
            
        # Use CLS token for prediction
        return self.head(x[:, 0])

print("FT-Transformer Model Defined.")

4.4 阶段四：FT-Transformer 深度优化 (H100)
FT-Transformer Model Defined.


In [9]:
# 4.4.1 H100 专属优化配置 - 数据加载
print("Preparing data for H100 optimization (Train + Test)...")

# Ensure we have the dataframes
if 'dae_df' not in locals():
    dae_df = pd.read_parquet('dae_features.parquet')
if 'knn_df' not in locals():
    knn_df = pd.read_parquet('knn_features.parquet')
    
# Load Test Features
if 'dae_test_df' not in locals():
    dae_test_df = pd.read_parquet('dae_features_test.parquet')
if 'knn_test_df' not in locals():
    knn_test_df = pd.read_parquet('knn_features_test.parquet')

# Combine all features: Raw + KNN + DAE
# Construct final feature matrix
X_final = pd.concat([train_df[feature_cols], knn_df, dae_df], axis=1).values
y_final = train_df[target_cols].values

X_final_test = pd.concat([test_df[feature_cols], knn_test_df, dae_test_df], axis=1).values

print(f"Final Input Shape (Train): {X_final.shape}")
print(f"Final Input Shape (Test): {X_final_test.shape}")

# Move to GPU directly (H100 optimization)
X_tensor = torch.tensor(X_final, dtype=torch.float32).to(DEVICE)
y_tensor = torch.tensor(y_final, dtype=torch.float32).to(DEVICE)
X_test_tensor = torch.tensor(X_final_test, dtype=torch.float32).to(DEVICE)

# Check for BF16 support
try:
    if torch.cuda.is_bf16_supported():
        dtype_train = torch.bfloat16
        print("✅ BF16 (Bfloat16) supported and enabled.")
    else:
        dtype_train = torch.float16
        print("⚠️ BF16 not supported, falling back to FP16.")
except:
    dtype_train = torch.float32
    print("⚠️ BF16 check failed, using FP32.")

Preparing data for H100 optimization (Train + Test)...
Final Input Shape (Train): (139392, 118)
Final Input Shape (Test): (34348, 118)
✅ BF16 (Bfloat16) supported and enabled.


In [16]:
import optuna
from optuna.trial import TrialState
import gc
import torch.nn as nn

print("4.4.3 Optuna 超参数搜索策略 (Competition Mode: Dual T4 GPU)")
print("=" * 80)

# Enable CuDNN benchmark for T4
torch.backends.cudnn.benchmark = True

def objective(trial):
    # Clear memory before each trial
    gc.collect()
    torch.cuda.empty_cache()

    # Hyperparameters Search Space
    # 降低 d_token 上限以防止 OOM，或者减小 Batch Size
    d_token = trial.suggest_categorical('d_token', [96, 192, 256]) # 移除 384 以防万一，或者保留但减小 Batch Size
    n_layers = trial.suggest_int('n_layers', 2, 4) 
    n_heads = 8 
    attention_dropout = trial.suggest_float('attention_dropout', 0.1, 0.3)
    ffn_dropout = trial.suggest_float('ffn_dropout', 0.1, 0.3)
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    d_ffn_factor = 1.33 
    
    # CV Loop (Competition: Use 3 folds for robust estimation)
    val_losses = []
    search_splits = splits[:1] 
    
    # 动态调整 Batch Size
    # 如果 d_token 较大，使用较小的 Batch Size
    if d_token >= 256:
        BATCH_SIZE = 1024
    else:
        BATCH_SIZE = 2048
        
    try:
        for fold_idx, (train_idx, val_idx) in enumerate(search_splits):
            # Model Initialization
            model = FTTransformer(
                n_num_features=X_final.shape[1],
                n_targets=3,
                d_token=d_token,
                n_layers=n_layers,
                n_heads=n_heads,
                d_ffn_factor=d_ffn_factor,
                attention_dropout=attention_dropout,
                ffn_dropout=ffn_dropout
            ).to(DEVICE)
            
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
            criterion = nn.L1Loss()
            
            # Data Slicing
            X_tr, y_tr = X_tensor[train_idx], y_tensor[train_idx]
            X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]
            
            # Training Loop
            model.train()
            n_batches = (len(X_tr) + BATCH_SIZE - 1) // BATCH_SIZE
            
            EPOCHS_SEARCH = 5
            
            for epoch in range(EPOCHS_SEARCH):
                perm = torch.randperm(len(X_tr), device=DEVICE)
                
                for i in range(n_batches):
                    idx = perm[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    batch_x, batch_y = X_tr[idx], y_tr[idx]
                    
                    with torch.cuda.amp.autocast(dtype=dtype_train):
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    
                # Pruning check
                if fold_idx == 0:
                    model.eval()
                    val_loss_epoch = 0.0
                    val_steps = 0
                    n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
                    
                    with torch.no_grad():
                        for i in range(n_val_batches):
                            batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                            batch_y = y_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                            with torch.cuda.amp.autocast(dtype=dtype_train):
                                outputs = model(batch_x)
                                loss = criterion(outputs, batch_y)
                            val_loss_epoch += loss.item()
                            val_steps += 1
                    val_loss_epoch /= val_steps
                    
                    trial.report(val_loss_epoch, epoch)
                    if trial.should_prune():
                        raise optuna.exceptions.TrialPruned()
                    model.train()
            
            # Final Validation
            model.eval()
            val_loss_fold = 0.0
            val_steps = 0
            n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
            with torch.no_grad():
                for i in range(n_val_batches):
                    batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    batch_y = y_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                    with torch.cuda.amp.autocast(dtype=dtype_train):
                        outputs = model(batch_x)
                        loss = criterion(outputs, batch_y)
                    val_loss_fold += loss.item()
                    val_steps += 1
            val_losses.append(val_loss_fold / val_steps)
            
            # Cleanup
            del model, optimizer, X_tr, y_tr, X_val, y_val
            gc.collect()
            torch.cuda.empty_cache()
            
    except torch.cuda.OutOfMemoryError:
        print(f"⚠️ Trial {trial.number} failed due to OOM. Pruning...")
        gc.collect()
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()
    except Exception as e:
        print(f"⚠️ Trial {trial.number} failed with error: {e}")
        raise e
        
    return sum(val_losses) / len(val_losses)

# Create Study
study = optuna.create_study(
    direction='minimize', 
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner()
)

print(f"Starting Optuna optimization on {torch.cuda.device_count()} GPUs...")
# Competition: Run more trials
study.optimize(objective, n_trials=10) 

print("Best params:", study.best_params)
print("Best value:", study.best_value)

[I 2025-12-26 10:02:45,011] A new study created in memory with name: no-name-8810e536-4eac-400e-b5b2-561792bdb5a5


4.4.3 Optuna 超参数搜索策略 (Competition Mode: Dual T4 GPU)
Starting Optuna optimization on 2 GPUs...


[I 2025-12-26 10:03:07,285] Trial 0 finished with value: 0.016428417370965082 and parameters: {'d_token': 96, 'n_layers': 3, 'attention_dropout': 0.20279515525047948, 'ffn_dropout': 0.24532275242109006, 'lr': 6.196193364069688e-05, 'weight_decay': 0.0005230266934891035}. Best is trial 0 with value: 0.016428417370965082.
[I 2025-12-26 10:03:32,789] Trial 1 finished with value: 0.01392276734923539 and parameters: {'d_token': 256, 'n_layers': 2, 'attention_dropout': 0.2211833642632761, 'ffn_dropout': 0.2290871749314641, 'lr': 3.30783817226068e-05, 'weight_decay': 4.911145308243903e-06}. Best is trial 1 with value: 0.01392276734923539.
[I 2025-12-26 10:03:41,258] Trial 2 pruned. 


⚠️ Trial 2 failed with error: 


[I 2025-12-26 10:03:47,088] Trial 3 pruned. 


...

⚠️ Trial 7 failed with error: 


[I 2025-12-26 10:04:58,182] Trial 8 pruned. 


⚠️ Trial 8 failed with error: 


[I 2025-12-26 10:05:06,557] Trial 9 pruned. 


⚠️ Trial 9 failed with error: 
Best params: {'d_token': 256, 'n_layers': 2, 'attention_dropout': 0.2211833642632761, 'ffn_dropout': 0.2290871749314641, 'lr': 3.30783817226068e-05, 'weight_decay': 4.911145308243903e-06}
Best value: 0.01392276734923539


In [17]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import gc

print("4.4.4 生成 FT-Transformer OOF 预测 (Competition Mode + Test Inference)")
print("=" * 80)

# Enable CuDNN benchmark for T4
torch.backends.cudnn.benchmark = True

# 1. 获取最佳参数
if 'study' in locals() and len(study.trials) > 0:
    best_params = study.best_params
    print(f"Using Optuna best params: {best_params}")
else:
    # Fallback params
    best_params = {
        'd_token': 256,
        'n_layers': 2,
        'attention_dropout': 0.2211833642632761,
        'ffn_dropout': 0.2290871749314641,
        'lr': 3.30783817226068e-05,
        'weight_decay': 4.911145308243903e-06
    }
    print(f"Using default params: {best_params}")

# 2. 初始化 OOF 容器
oof_ftt = np.zeros(y_final.shape)
test_preds_ftt = np.zeros((len(X_final_test), 3)) 

# 3. 全量 CV 训练
print(f"Starting Full Cross-Validation on {len(splits)} folds...")

for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n--- Fold {fold + 1}/{len(splits)} ---")
    
    # Data Preparation
    X_tr, y_tr = X_tensor[train_idx], y_tensor[train_idx]
    X_val, y_val = X_tensor[val_idx], y_tensor[val_idx]
    
    # Model Initialization
    model = FTTransformer(
        n_num_features=X_final.shape[1],
        n_targets=3,
        d_token=best_params.get('d_token', 192),
        n_layers=best_params.get('n_layers', 4),
        n_heads=8,
        d_ffn_factor=1.33,
        attention_dropout=best_params.get('attention_dropout', 0.2),
        ffn_dropout=best_params.get('ffn_dropout', 0.1)
    ).to(DEVICE)
    
    # Multi-GPU Support (DataParallel)
    if torch.cuda.device_count() > 1:
        print(f"  Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)
    else:
        # Compile (H100 optimization / Single GPU)
        try:
            print("  Using torch.compile (Single GPU)")
            model = torch.compile(model)
        except:
            pass
    
    optimizer = optim.AdamW(model.parameters(), lr=best_params.get('lr', 1e-4), weight_decay=best_params.get('weight_decay', 1e-5))
    criterion = nn.L1Loss()
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=best_params.get('lr', 1e-4)*10, steps_per_epoch=(len(X_tr)//2048)+1, epochs=30)

    # Training Loop
    model.train()
    BATCH_SIZE = 2048
    n_batches = (len(X_tr) + BATCH_SIZE - 1) // BATCH_SIZE
    EPOCHS = 30 # Competition standard
    
    for epoch in range(EPOCHS):
        perm = torch.randperm(len(X_tr), device=DEVICE)
        for i in range(n_batches):
            idx = perm[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            batch_x, batch_y = X_tr[idx], y_tr[idx]
            
            with torch.cuda.amp.autocast(dtype=dtype_train):
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            
    # Inference on Validation (OOF)
    model.eval()
    val_preds_fold = []
    n_val_batches = (len(X_val) + BATCH_SIZE - 1) // BATCH_SIZE
    
    with torch.no_grad():
        for i in range(n_val_batches):
            batch_x = X_val[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            with torch.cuda.amp.autocast(dtype=dtype_train):
                preds = model(batch_x)
            val_preds_fold.append(preds.cpu().numpy())
            
    oof_ftt[val_idx] = np.vstack(val_preds_fold)
    
    # Inference on Test (Accumulate)
    test_preds_fold = []
    n_test_batches = (len(X_test_tensor) + BATCH_SIZE - 1) // BATCH_SIZE
    
    with torch.no_grad():
        for i in range(n_test_batches):
            batch_x = X_test_tensor[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            with torch.cuda.amp.autocast(dtype=dtype_train):
                preds = model(batch_x)
            test_preds_fold.append(preds.cpu().numpy())
            
    test_preds_ftt += np.vstack(test_preds_fold) / len(splits)
    
    # Cleanup
    del model, optimizer, scheduler, X_tr, y_tr, X_val, y_val
    gc.collect()
    torch.cuda.empty_cache()

print("FT-Transformer OOF & Test generation complete.")

4.4.4 生成 FT-Transformer OOF 预测 (Competition Mode + Test Inference)
Using Optuna best params: {'d_token': 256, 'n_layers': 2, 'attention_dropout': 0.2211833642632761, 'ffn_dropout': 0.2290871749314641, 'lr': 3.30783817226068e-05, 'weight_decay': 4.911145308243903e-06}
Starting Full Cross-Validation on 5 folds...

--- Fold 1/5 ---
  Using 2 GPUs with DataParallel

--- Fold 2/5 ---
  Using 2 GPUs with DataParallel

--- Fold 3/5 ---
  Using 2 GPUs with DataParallel

--- Fold 4/5 ---
  Using 2 GPUs with DataParallel

--- Fold 5/5 ---
  Using 2 GPUs with DataParallel
FT-Transformer OOF & Test generation complete.


In [18]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

print("4.5 阶段五：GBDT 多目标回归 (XGBoost - High Noise Regime + Test Inference)")
print("=" * 80)

# 4.5.2 特征输入: 使用全量特征 (X_final)
print(f"GBDT Input Shape: {X_final.shape}")

# 初始化 OOF 容器
oof_xgb = np.zeros(y_final.shape)
test_preds_xgb = np.zeros((len(X_final_test), 3))

# 4.5.3 训练配置 (针对高噪声金融数据的极端防御配置)
xgb_params = {
    'tree_method': 'hist',
    'device': 'cuda',
    'objective': 'reg:squarederror', 
    'eval_metric': 'mae',
    
    # --- 核心抗噪参数 ---
    'learning_rate': 0.005, # 极低的学习率，步步为营
    'max_depth': 3,         # 极浅的树，只学大规律，不扣细节
    'min_child_weight': 200,# 一个叶子必须包含 200+ 样本，防止拟合个例
    'gamma': 0.1,           # 分裂所需的最小 Loss 减少量
    'base_score': 0.0,      # 收益率均值接近 0，显式指定
    
    # --- 正则化 ---
    'colsample_bytree': 0.5, # 每次只看一半特征
    'subsample': 0.6,        # 每次只看 60% 数据
    'reg_lambda': 20.0,      # 极强的 L2 正则
    'reg_alpha': 5.0,        # 增加 L1 正则，进行特征选择
    
    'n_estimators': 8000,    # 配合低学习率，增加树的数量
    'early_stopping_rounds': 200,
    'n_jobs': -1
}

targets = ['short', 'medium', 'long']

for i, target_name in enumerate(targets):
    print(f"\nTraining XGBoost for Target: {target_name}")
    
    for fold, (train_idx, val_idx) in enumerate(splits):
        # Prepare Data
        X_tr, y_tr = X_final[train_idx], y_final[train_idx, i]
        X_val, y_val = X_final[val_idx], y_final[val_idx, i]
        
        # Train
        model = xgb.XGBRegressor(**xgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=1000 # 减少打印频率
        )
        
        # Predict OOF
        val_preds = model.predict(X_val)
        oof_xgb[val_idx, i] = val_preds
        
        # Predict Test (Accumulate)
        test_preds_xgb[:, i] += model.predict(X_final_test) / len(splits)
        
        # Check if model learned anything (Best iteration > 0)
        if model.best_iteration < 5:
            print(f"  ⚠️ Fold {fold+1}: Model failed to learn (Best Iter: {model.best_iteration}). Signal is too weak.")
        else:
            print(f"  ✅ Fold {fold+1}: Learned {model.best_iteration} trees. Best MAE: {model.best_score:.5f}")

print("\nXGBoost Training Complete.")
print("OOF MAE per target (Purged CV):")
mae_short = mean_absolute_error(y_final[:, 0], oof_xgb[:, 0])
mae_medium = mean_absolute_error(y_final[:, 1], oof_xgb[:, 1])
mae_long = mean_absolute_error(y_final[:, 2], oof_xgb[:, 2])
print(f"Short: {mae_short:.6f}")
print(f"Medium: {mae_medium:.6f}")
print(f"Long: {mae_long:.6f}")

4.5 阶段五：GBDT 多目标回归 (XGBoost - High Noise Regime + Test Inference)
GBDT Input Shape: (139392, 118)

Training XGBoost for Target: short
[0]	validation_0-mae:0.00388
[200]	validation_0-mae:0.00388
  ⚠️ Fold 1: Model failed to learn (Best Iter: 0). Signal is too weak.
[0]	validation_0-mae:0.00412
[199]	validation_0-mae:0.00412
  ⚠️ Fold 2: Model failed to learn (Best Iter: 0). Signal is too weak.
[0]	validation_0-mae:0.00407
[199]	validation_0-mae:0.00407
  ⚠️ Fold 3: Model failed to learn (Best Iter: 0). Signal is too weak.
[0]	validation_0-mae:0.00357
[200]	validation_0-mae:0.00357
  ⚠️ Fold 4: Model failed to learn (Best Iter: 0). Signal is too weak.
[0]	validation_0-mae:0.00286
[371]	validation_0-mae:0.00286
  ✅ Fold 5: Learned 171 trees. Best MAE: 0.00286

Training XGBoost for Target: medium
[0]	validation_0-mae:0.01003
[465]	validation_0-mae:0.01002
  ✅ Fold 1: Learned 266 trees. Best MAE: 0.01002
[0]	validation_0-mae:0.01091
[1000]	validation_0-mae:0.01090
[2000]	validation_0-mae:0.

In [19]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

print("4.6 阶段六：集成与 Stacking (Ridge Regression) + 生成提交")
print("=" * 80)

# 0. 安全检查与保存
if 'oof_ftt' not in locals():
    print("⚠️ Warning: 'oof_ftt' not found. Using zeros for FTT.")
    oof_ftt = np.zeros_like(oof_xgb)
    test_preds_ftt = np.zeros_like(test_preds_xgb)

# 保存中间结果
np.savez('oof_predictions.npz', oof_ftt=oof_ftt, oof_xgb=oof_xgb, y_final=y_final)
print("✅ OOF predictions saved to 'oof_predictions.npz'")

# 1. 准备 Stacking
stacking_weights = {}
final_oof_preds = np.zeros(y_final.shape)
final_test_preds = np.zeros((len(test_preds_xgb), 3))
targets = ['short', 'medium', 'long']

print(f"\n{'Target':<10} | {'FTT Weight':<12} | {'XGB Weight':<12} | {'Bias':<10}")
print("-" * 54)

for i, target_name in enumerate(targets):
    # Stacking Input (Train): [FTT_pred, XGB_pred]
    X_meta = np.column_stack([oof_ftt[:, i], oof_xgb[:, i]])
    y_meta = y_final[:, i]
    
    # Meta-Model: Ridge Regression
    meta_model = Ridge(alpha=10.0)
    meta_model.fit(X_meta, y_meta)
    
    weights = meta_model.coef_
    intercept = meta_model.intercept_
    
    w_sum = np.sum(np.abs(weights)) + 1e-9
    w_norm = weights / w_sum
    
    print(f"{target_name:<10} | {weights[0]:.4f} ({w_norm[0]:.2f}) | {weights[1]:.4f} ({w_norm[1]:.2f}) | {intercept:.4f}")
    
    stacking_weights[target_name] = meta_model
    final_oof_preds[:, i] = meta_model.predict(X_meta)
    
    # Stacking Input (Test): [FTT_pred, XGB_pred]
    X_meta_test = np.column_stack([test_preds_ftt[:, i], test_preds_xgb[:, i]])
    final_test_preds[:, i] = meta_model.predict(X_meta_test)

# Calculate Final Weighted MAE
final_mae_short = mean_absolute_error(y_final[:, 0], final_oof_preds[:, 0])
final_mae_medium = mean_absolute_error(y_final[:, 1], final_oof_preds[:, 1])
final_mae_long = mean_absolute_error(y_final[:, 2], final_oof_preds[:, 2])

if 'TARGET_WEIGHTS' not in locals():
    TARGET_WEIGHTS = {'short': 0.3, 'medium': 0.3, 'long': 0.4}

final_weighted_mae = (
    TARGET_WEIGHTS['short'] * final_mae_short +
    TARGET_WEIGHTS['medium'] * final_mae_medium +
    TARGET_WEIGHTS['long'] * final_mae_long
)

print("\n" + "="*80)
print("Final Ensemble Summary")
print("="*80)
print(f"Short MAE:  {final_mae_short:.6f}")
print(f"Medium MAE: {final_mae_medium:.6f}")
print(f"Long MAE:   {final_mae_long:.6f}")
print(f"Weighted MAE: {final_weighted_mae:.6f}")
print("="*80)

# 2. 生成提交文件
print("\nGenerating Submission File...")
submission = pd.DataFrame({
    'id': test_df['id'] if 'id' in test_df.columns else test_df.index,
    'target_short': final_test_preds[:, 0],
    'target_medium': final_test_preds[:, 1],
    'target_long': final_test_preds[:, 2]
})

submission_file = 'submission.csv'
submission.to_csv(submission_file, index=False)
print(f"✅ Submission saved to: {submission_file}")
print(submission.head())

4.6 阶段六：集成与 Stacking (Ridge Regression) + 生成提交
✅ OOF predictions saved to 'oof_predictions.npz'

Target     | FTT Weight   | XGB Weight   | Bias      
------------------------------------------------------
short      | 0.0015 (1.00) | 0.0000 (0.00) | -0.0000
medium     | 0.0286 (0.99) | -0.0004 (-0.01) | -0.0002
long       | 0.0707 (0.68) | 0.0330 (0.32) | -0.0012

Final Ensemble Summary
Short MAE:  0.003775
Medium MAE: 0.009986
Long MAE:   0.021915
Weighted MAE: 0.009267

Generating Submission File...
✅ Submission saved to: submission.csv
   id  target_short  target_medium  target_long
0   0     -0.000033      -0.000104     0.000773
1   1     -0.000033      -0.000102     0.000661
2   2     -0.000033      -0.000097     0.000654
3   3     -0.000033      -0.000100     0.000670
4   4     -0.000033      -0.000112     0.000668


In [ ]:
split_idx = int(len(train_df) * 0.8)

X_train = train_df[feature_cols].iloc[:split_idx].values
y_train = train_df[target_cols].iloc[:split_idx].values
X_val = train_df[feature_cols].iloc[split_idx:].values
y_val = train_df[target_cols].iloc[split_idx:].values
X_test = test_df[feature_cols].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
print("="*80)
print("Training XGBoost Models")
print("="*80)

xgb_models = {}
xgb_val_predictions = {}

for i, target_name in enumerate(['short', 'medium', 'long']):
    print(f"\n--- Training XGBoost for target_{target_name} ---")
    
    # XGBoost parameters with early stopping
    xgb_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'max_depth': 6,#4
        'learning_rate': 0.01,
        'n_estimators': 500,#100
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'random_state': 42,
        'eval_metric': 'mae',
        'early_stopping_rounds': 50
    }
    
    model = xgb.XGBRegressor(**xgb_params)
    
    # Train
    model.fit(
        X_train, y_train[:, i],
        eval_set=[(X_val, y_val[:, i])],
        verbose=100
    )
    
    # Predict on validation set
    val_pred = model.predict(X_val)
    val_mae = mean_absolute_error(y_val[:, i], val_pred)
    
    print(f"Best iteration: {model.best_iteration}")
    print(f"Validation MAE: {val_mae:.6f}")
    
    # Store model and predictions
    xgb_models[target_name] = model
    xgb_val_predictions[target_name] = val_pred

# Calculate weighted MAE for XGBoost
xgb_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(y_val[:, 2], xgb_val_predictions['long'])
)

print("\n" + "="*80)
print("XGBoost Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(y_val[:, 0], xgb_val_predictions['short']):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(y_val[:, 1], xgb_val_predictions['medium']):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(y_val[:, 2], xgb_val_predictions['long']):.6f} (weight: 0.2)")
print(f"Weighted MAE: {xgb_weighted_mae:.6f}")
print("="*80)

In [ ]:
# Standardize features for neural network (don't scale targets)
print("Standardizing features...")
from sklearn.preprocessing import RobustScaler

scaler_X = RobustScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

print("Standardization complete")

class MultiTargetDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y) if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

# Create datasets (use original y_train, y_val - no scaling)
train_dataset = MultiTargetDataset(X_train_scaled, y_train)
val_dataset = MultiTargetDataset(X_val_scaled, y_val)
test_dataset = MultiTargetDataset(X_test_scaled)

# Create dataloaders
BATCH_SIZE = 512*8*8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

class MLPMultiTarget(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64], num_targets=3, dropout=0.2):
        super(MLPMultiTarget, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        
        self.shared_layers = nn.Sequential(*layers)
        
        # Output layer: predict all 3 targets simultaneously
        self.output_layer = nn.Linear(prev_dim, num_targets)
    
    def forward(self, x):
        x = self.shared_layers(x)
        return self.output_layer(x)

# Initialize model
INPUT_DIM = X_train_scaled.shape[1]
mlp_model = MLPMultiTarget(input_dim=INPUT_DIM).to(DEVICE)

print(f"MLP Model:")
print(mlp_model)
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 2:
                X_batch, y_batch = batch
                X_batch = X_batch.to(device)
                
                outputs = model(X_batch)
                
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(y_batch.numpy())
            else:
                X_batch = batch.to(device)
                outputs = model(X_batch)
                all_preds.append(outputs.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels) if len(all_labels) > 0 else None
    
    return all_preds, all_labels

print("Training functions defined")

In [ ]:
print("Training MLP model...")
print("="*80)

criterion = nn.L1Loss()  # Use MAE loss directly
optimizer = optim.AdamW(mlp_model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

EPOCHS = 20
best_weighted_mae = float('inf')
patience = 5
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss = train_epoch(mlp_model, train_loader, criterion, optimizer, DEVICE)
    val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)
    
    # Calculate MAE for each target
    mae_short = mean_absolute_error(val_labels[:, 0], val_preds[:, 0])
    mae_medium = mean_absolute_error(val_labels[:, 1], val_preds[:, 1])
    mae_long = mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
    
    # Calculate weighted MAE
    weighted_mae = (
        TARGET_WEIGHTS['short'] * mae_short +
        TARGET_WEIGHTS['medium'] * mae_medium +
        TARGET_WEIGHTS['long'] * mae_long
    )
    
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(weighted_mae)
    new_lr = optimizer.param_groups[0]['lr']
    
    if new_lr != old_lr:
        print(f"  Learning rate reduced: {old_lr:.6f} -> {new_lr:.6f}")
    
    if weighted_mae < best_weighted_mae:
        best_weighted_mae = weighted_mae
        patience_counter = 0
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}:")
        print(f"  Train Loss: {train_loss:.6f}")
        print(f"  Val Weighted MAE: {weighted_mae:.6f} (Best: {best_weighted_mae:.6f})")
        print(f"    Short: {mae_short:.6f}, Medium: {mae_medium:.6f}, Long: {mae_long:.6f}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nMLP training complete")
print(f"Best validation Weighted MAE: {best_weighted_mae:.6f}")

In [ ]:
val_preds, val_labels = eval_epoch(mlp_model, val_loader, DEVICE)

mlp_weighted_mae = (
    TARGET_WEIGHTS['short'] * mean_absolute_error(val_labels[:, 0], val_preds[:, 0]) +
    TARGET_WEIGHTS['medium'] * mean_absolute_error(val_labels[:, 1], val_preds[:, 1]) +
    TARGET_WEIGHTS['long'] * mean_absolute_error(val_labels[:, 2], val_preds[:, 2])
)

print("="*80)
print("MLP Summary")
print("="*80)
print(f"Short MAE:  {mean_absolute_error(val_labels[:, 0], val_preds[:, 0]):.6f} (weight: 0.5)")
print(f"Medium MAE: {mean_absolute_error(val_labels[:, 1], val_preds[:, 1]):.6f} (weight: 0.3)")
print(f"Long MAE:   {mean_absolute_error(val_labels[:, 2], val_preds[:, 2]):.6f} (weight: 0.2)")
print(f"Weighted MAE: {mlp_weighted_mae:.6f}")
print("="*80)

In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['XGBoost (3 models)', 'MLP (multi-target)'],
    'Weighted MAE': [xgb_weighted_mae, mlp_weighted_mae]
})

print(comparison.to_string(index=False))

best_model = 'XGBoost' if xgb_weighted_mae < mlp_weighted_mae else 'MLP'
print(f"\nBest model: {best_model}")

In [ ]:
print("Generating XGBoost predictions...")
test_pred_xgb_short = xgb_models['short'].predict(X_test)
test_pred_xgb_medium = xgb_models['medium'].predict(X_test)
test_pred_xgb_long = xgb_models['long'].predict(X_test)

submission_xgb = pd.DataFrame({
    'id': test_df['id'],
    'target_short': test_pred_xgb_short,
    'target_medium': test_pred_xgb_medium,
    'target_long': test_pred_xgb_long
})
submission_xgb.to_csv('submission_xgb.csv', index=False)
print(f"✅ XGBoost submission saved: submission_xgb.csv")

# print("\nGenerating MLP predictions...")
# test_preds_mlp, _ = eval_epoch(mlp_model, test_loader, DEVICE)
# submission_mlp = pd.DataFrame({
#     'id': test_df['id'],
#     'target_short': test_preds_mlp[:, 0],
#     'target_medium': test_preds_mlp[:, 1],
#     'target_long': test_preds_mlp[:, 2]
# })
# submission_mlp.to_csv('submission.csv', index=False)
# print(f"✅ MLP submission saved: submission_mlp.csv")

In [ ]:
import os
print(os.getcwd())
files = os.listdir('.')     # 列出当前目录下所有文件和文件夹
print(files)

In [ ]:
from IPython.display import FileLink
FileLink(r'submission.csv')